In [7]:
import os
import numpy as np
import pandas as pd

os.makedirs("../data/processed", exist_ok=True)

# 1. 원본 CSV 불러오기
raw_df = pd.read_csv("../data/raw/seoul_population.csv", encoding="utf-8-sig")

# 2. 두 번째 행(1번 인덱스)에 있는 실제 연령대 이름을 컬럼명으로 재설정
age_headers = raw_df.iloc[1].tolist()
df_clean = raw_df.iloc[2:].copy()  # 실제 데이터(2번 행부터)만 추출
df_clean.columns = ["시도", "자치구", "성별", "총인구"] + age_headers[4:]

# 3. '소계'(서울시 전체 합계) 제외 및 성별이 '계'(남녀 합계)인 행만 필터링 -> 서울 25개 자치구 추출
df_clean = df_clean[
    (df_clean["자치구"] != "소계") & (df_clean["성별"] == "계")
].copy()

# 4. 65세 이상 고령인구 컬럼 합산 및 숫자형 변환
elderly_cols = [
    "65~69세",
    "70~74세",
    "75~79세",
    "80~84세",
    "85~89세",
    "90~94세",
    "95~99세",
    "100세 이상+",
]

for col in ["총인구"] + elderly_cols:
  df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce").fillna(0)

df_clean["elderly_pop"] = df_clean[elderly_cols].sum(axis=1)

# 5. 프로젝트 표준 데이터프레임 생성
df = pd.DataFrame()
df["admin_code"] = [f"11{i:03d}" for i in range(1, len(df_clean) + 1)]
df["region_name"] = df_clean["자치구"].values

# 실제 데이터 기반 고령인구 비율(%) 계산 (예: 종로구 21.81%, 중구 22.42%)
df["elderly_ratio"] = (
    df_clean["elderly_pop"] / df_clean["총인구"] * 100
).values

# 나머지 변수는 25개 자치구 길이에 맞춰 시뮬레이션 데이터 생성
np.random.seed(42)
n_samples = len(df)  # 25개 자치구
df["single_household_ratio"] = np.random.uniform(20.0, 50.0, n_samples)
df["welfare_facilities"] = np.random.randint(2, 15, n_samples)
df["care_workers"] = np.random.randint(15, 60, n_samples)
df["annual_budget"] = np.random.uniform(5.0, 25.0, n_samples)

# 6. 위험 지수(risk_score) 산출
df["risk_score"] = (
    df["elderly_ratio"] * 1.2
    + df["single_household_ratio"] * 0.8
    - df["welfare_facilities"] * 1.2
    - df["care_workers"] * 0.3
    - df["annual_budget"] * 0.8
    + np.random.normal(0, 2, n_samples)
).clip(10, 95)

# 7. 최종 전처리 데이터 저장
df.to_csv("../data/processed/merged_care_dataset.csv", index=False)
print(
    f"✅ 서울시 {n_samples}개 자치구 실제 인구 데이터 전처리 및 저장 완료!"
)
display(df.head(10))

✅ 서울시 25개 자치구 실제 인구 데이터 전처리 및 저장 완료!


,admin_code,region_name,elderly_ratio,single_household_ratio,welfare_facilities,care_workers,annual_budget,risk_score
0,11001,종로구,21.807928,31.236204,4,21,8.919657,31.964402
1,11002,중구,22.418318,48.521429,13,58,5.904546,27.624170
2,11003,용산구,19.348170,41.959818,8,22,11.506607,29.167703
3,11004,성동구,19.001338,37.959755,5,49,12.773546,19.858159
4,11005,광진구,18.342812,24.680559,10,28,10.426981,14.639289
5,11006,동대문구,20.715157,24.679836,4,31,21.574750,15.954737
6,11007,중랑구,22.672430,21.742508,6,50,12.135067,12.548849
7,11008,성북구,20.217188,45.985284,4,54,10.618690,33.560966
8,11009,강북구,25.931391,38.033450,8,18,15.853922,34.584564
9,11010,도봉구,25.286954,41.242177,6,16,7.818484,43.793059
